## Задание 2 вариант 1

Нам попался избитый жизнью датасет про ирисы

In [194]:
import numpy as np
import pandas as pd
from scipy.stats import f

In [195]:
df = pd.read_csv('iris.csv', encoding='utf-8')
print(df.head())

   Sepal.Length  Sepal.Width  Petal.Length  Petal.Width Species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa


##### Добавим поле 'Area' и в глазами посмотрим на общее среднее и среднее в каждой группе

In [196]:
df['Area'] = (
    df['Sepal.Length'] * df['Sepal.Width'] +
    df['Petal.Length'] * df['Petal.Width']
)

groups = {
    name: group['Area'].values
    for name, group in df.groupby('Species')
}

all_values = df['Area'].values
all_mean = np.mean(all_values)
J = len(groups)
I = len(all_values)

WIDTH = 32
CUT = 4

print(
    f'General',
    '-' * WIDTH,
    f'Уровень фактора: {J:>15}',
    f'Всего наблюений: {I:>15}',
    f'Среднее всех: {all_mean:>{18}.{CUT}f}',
    sep='\n'
)

print(
    '-' * WIDTH,
    'Average area by group',
    '-' * WIDTH,
    sep='\n'
)
for group_name, group_values in groups.items():
    print(f'\'{group_name}\': {np.mean(group_values):>{WIDTH - len(group_name) - CUT}.{CUT}f}')

General
--------------------------------
Уровень фактора:               3
Всего наблюений:             150
Среднее всех:            23.6169
--------------------------------
Average area by group
--------------------------------
'setosa':                17.6234
'versicolor':            22.2466
'virginica':             30.9808


##### Аккуратно по формулам с лекции посчитаем F статистику и сравнимм ее с табличными данными. Гипотезы следующие:

##### 1. $H_0: \mu_\text{setosa} = \mu_\text{versicolor} = \mu_\text{virginica}$

##### 2. $H_1 =\neg H_0$

In [197]:
S_B = sum(
    len(v) * (np.mean(v) - all_mean) ** 2
    for v in groups.values()
)
S_W = sum(
    np.sum((v - np.mean(v)) ** 2)
    for v in groups.values()
)

df_B = J - 1
df_W = I - J
F = (S_B * df_W) / (S_W * df_B)


alpha = 0.05
F_crit = f.ppf(1 - alpha, df_B, df_W)
p_value = 1 - f.cdf(F, df_B, df_W)


print(
    f'Statistics',
    '-' * WIDTH,
    f'p-value: {p_value:>{23}.{CUT}f}',
    f'F: {F:>{29}.{CUT}f}',
    f'F critical: {F_crit:>{20}.{CUT}f}',
    sep='\n'
)

print(
    '-' * WIDTH,
    'Result',
    '-' * WIDTH,
    sep='\n'
)

if F > F_crit:
    print('Отвергаем H0 в пользу H1')
else:
    print('Не отвергаем H0')

Statistics
--------------------------------
p-value:                  0.0000
F:                      133.2972
F critical:               3.0576
--------------------------------
Result
--------------------------------
Отвергаем H0 в пользу H1
